# 강화학습 7강 · LINK-01 걷기를 PPO로 처음부터

6강에서는 `gym.make` 로 세상을 열고, PPO 로 배우고, 허깅페이스에 올렸습니다. 오늘도 똑같은 순서입니다.
다른 점은 세상이 **우리가 만든 두 발 로봇 LINK-01** 이고, 관절이 **14개**라는 것. 그리고 학습 코드는 **여러분이 직접** 씁니다.

| 순서 | 하는 일 |
|---|---|
| CELL 01 | 설치 · 우리 시뮬레이터 `link01-gym` |
| CELL 02 | `gym.make("Link01Walk-v0")` |
| CELL 03 | 학습 전 — 바로 넘어집니다 (Humanoid 처럼) |
| CELL 04 | 보상과 끝 — 세상이 이미 줍니다 |
| CELL 05 | ✏️ **PPO 학습 코드 — 직접 작성** |
| CELL 06 | 시험 + 리플레이 영상 |
| CELL 07 | 🤗 허깅페이스에 올리기 (내 토큰) |

## CELL 01 · 설치

강의 페이지에서 받은 **`link01-gym-ppo7.zip`** 을 코랩 왼쪽 📁 에 올리고 실행하세요. (내 컴퓨터에서도 같습니다.)

In [ ]:
import os, sys, subprocess, time, zipfile
if 'google.colab' in sys.modules:
    os.environ['MUJOCO_GL'] = 'egl'          # 코랩: 화면 없이 로봇 그리기
    subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null 2>&1', shell=True)   # 한글 글꼴
Z = 'link01-gym-ppo7.zip'
if not os.path.isdir('link01-gym-ppo7') and not os.path.isdir('link01-gym'):
    assert os.path.exists(Z), '왼쪽 📁 에 link01-gym-ppo7.zip 을 먼저 올려 주세요.'
    # 업로드가 끝날 때까지 기다리기 (올리는 중에 실행하면 파일이 반만 있어요)
    for _ in range(120):
        s1 = os.path.getsize(Z); time.sleep(1); s2 = os.path.getsize(Z)
        if s1 == s2 and zipfile.is_zipfile(Z): break
        print('업로드 기다리는 중…', s2 // 1024, 'KB')
    assert zipfile.is_zipfile(Z), 'zip 이 다 올라가지 않았어요. 왼쪽 📁 에서 파일을 지우고 다시 올린 뒤 이 칸을 다시 실행하세요.'
    zipfile.ZipFile(Z).extractall('.')
if os.path.isdir('link01-gym-ppo7'):
    os.chdir('link01-gym-ppo7')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', './link01-gym[train]'], check=True)
print('준비 완료 · 지금 폴더:', os.getcwd())

## CELL 02 · 세상 열기 — `gym.make`

OpenAI Gym 은 지금 **Gymnasium** 이 이어서 관리합니다. `import gymnasium as gym` 이면 6강과 같은 모양 그대로입니다.

- 관측 **51개**: 몸 기울기 3 · 회전 속도 3 · 관절 각도 14 · 관절 속도 14 · 직전 행동 14 · 명령 3 (앞으로 몇 m/s)
- 행동 **14개**: 다리 10 + 머리 4 관절, 기본 자세에서 각자 −1 ~ +1 (= ±0.18 rad)
- 0.02초마다 한 번 결정 (1초에 50번), 한 판 최대 1,000번 = 20초

In [ ]:
import gymnasium as gym, link01_gym, numpy as np
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from gymnasium.utils.env_checker import check_env
글꼴경로 = next((p for p in ['/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf',
                            '/System/Library/Fonts/Supplemental/AppleGothic.ttf'] if os.path.exists(p)), None)
if 글꼴경로:
    fm.fontManager.addfont(글꼴경로); plt.rcParams['font.family'] = fm.FontProperties(fname=글꼴경로).get_name()
env = gym.make('Link01Walk-v0', render_mode='rgb_array')
obs, info = env.reset(seed=0)
print('관측', obs.shape, '· 행동', env.action_space.shape, '· 이번 판 명령 속도', round(float(info['command'][0]), 3), 'm/s')
check_env(env.unwrapped, skip_render_check=True); print('check_env 통과')
plt.figure(figsize=(6, 4.5)); plt.imshow(env.render()); plt.axis('off'); plt.show()

## CELL 03 · 학습 전 — 바로 넘어집니다

6강 마지막에 본 Humanoid 처럼, 아무것도 배우지 않은 두뇌(무작위 행동)는 몇 초 못 버팁니다.

In [ ]:
from base64 import b64encode
from IPython.display import HTML, display
from eval_walk import evaluate
def show(path, w=560):
    display(HTML(f'<video width={w} controls autoplay loop muted src="data:video/mp4;base64,{b64encode(open(path, "rb").read()).decode()}"></video>'))

e = gym.make('Link01Walk-v0'); o, _ = e.reset(seed=1); 걸음 = 0
while True:
    o, r, 끝, 잘림, 정보 = e.step(e.action_space.sample()); 걸음 += 1
    if 끝 or 잘림: break
print(f'무작위 두뇌: {걸음}걸음 = {걸음 * 0.02:.1f}초 만에 {"넘어짐" if 끝 else "시간 끝"}')
rng = np.random.default_rng(0)
m0 = evaluate(lambda o: rng.uniform(-1, 1, 14), 'runs/before', episodes=1, seconds=10, label='LINK-01 · 학습 전 (무작위)')
print(f"10초 동안 넘어짐 {m0['episodes_detail'][0]['falls']}번")
show('runs/before/replay.mp4')

## CELL 04 · 보상과 끝 — 세상이 이미 줍니다

`env.step(행동)` 이 돌려주는 `보상` 과 `끝(terminated)` 은 이미 만들어 두었습니다. 여러분은 **두뇌만** 만들면 됩니다.

| 이름 | 0.02초마다 | 뜻 |
|---|---|---|
| track | + 최대 2 | 명령 속도에 맞춰 앞으로 |
| progress | + 최대 1.2 | 앞으로 갈수록 + (서 있으면 0) |
| alive | + 0.2 | 안 넘어지고 있음 |
| upright · side | − | 몸이 기울거나, 옆으로 새거나, 돌면 |
| air_time | + | 발을 들어 진짜 걸음을 디디면 |
| energy · smooth | − | 힘을 많이 쓰거나 떨면 |
| **fall** | **−10, 판 끝** | 몸 높이 0.10 m 아래 또는 크게 기울면 |

> 그래서 「가만히 서 있기」 는 0.6점 정도, 「명령대로 걷기」 는 2.5점 이상을 받습니다. 넘어지면 그 판이 끝나니 점수를 더 못 받습니다.

In [ ]:
e = gym.make('Link01Walk-v0'); o, _ = e.reset(seed=0)
o, r, 끝, 잘림, 정보 = e.step(np.zeros(14))
print('보상', round(r, 3), '=', {k: round(v, 3) for k, v in 정보['terms'].items()})

## CELL 05 · ✏️ PPO 학습 — 직접 작성

아래 칸을 채우세요. 다 채우면 **세 가지**만 남기면 됩니다. 그러면 CELL 06 · 07 이 그대로 돌아갑니다.

| 남길 것 | 모양 |
|---|---|
| `행동` | 함수: 관측 51개(numpy) → 행동 14개(numpy) |
| `모델파일들` | 허깅페이스에 올릴 두뇌 파일 경로 리스트 (예: `['my_walk/model.zip', 'my_walk/vecnormalize.pkl']`) |
| `학습정보` | dict: `steps`(학습 걸음), `minutes`(걸린 분), 그 밖에 바꾼 숫자 |

**힌트**
1. **6강 PPO 그대로** — 6강 CELL 08 의 `PPO학습(세상만들기, 총걸음, …)` 은 행동이 연속(Box)이면 알아서 「평균 · 흔들림」 두뇌로 바꿉니다. `세상만들기 = lambda: gym.make('Link01Walk-v0')` 를 넣으면 됩니다. 결과 두뇌가 `뇌` 라면 `행동 = lambda o: 뇌.분포(torch.as_tensor(o, dtype=torch.float32)).mean.detach().numpy()`.
2. **라이브러리로** — `stable-baselines3` 의 `PPO('MlpPolicy', …)`. 세상 여러 개는 `make_vec_env('link01_gym:Link01Walk-v0', n_envs=…, vec_env_cls=SubprocVecEnv)` — 이름 앞의 `link01_gym:` 은 따로 도는 세상들도 우리 시뮬레이터를 불러오게 하는 표시입니다.
3. **관측 정규화가 중요합니다** — 관측 51개의 크기가 제각각입니다. SB3 는 `VecNormalize`, 직접 짠다면 평균 · 표준편차로 나눠 주세요. (정규화 값도 같이 저장해야 나중에 다시 걷습니다.)
4. 숫자는 6강과 같게 시작: 학습률 3e-4 · γ 0.99 · λ 0.95 · clip 0.2. 정책(행동) 쪽 은닉층은 **64, 64** 로 두세요 — 웹 플레이그라운드 두뇌와 같은 크기입니다. 가치 쪽은 크게 (예: 256, 256) 해도 됩니다.
5. 처음엔 20만 걸음으로 끝까지 도는지만 확인 → 그다음 200만 걸음 이상. 걸음이 쌓일수록 「판 평균 보상」 과 「판 길이」 가 올라가는지 보세요.
6. 막히면: 보상이 오르는데 앞으로 안 가면 서 있기만 배운 것 → 더 오래 학습하거나 `gym.make('Link01WalkGuided-v0')` (우리 참고 걸음의 리듬을 따라 하면 + 보상) 로 바꿔 보세요.

In [ ]:
import os, time
import torch
import gymnasium as gym
import link01_gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize

# ================= ✏️ PPO 학습 구현 (CONNECT AI LAB 7강) =================
# 1) 학습 환경 및 하이퍼파라미터 설정
환경이름 = 'Link01WalkGuided-v0'  # 리듬 가이드 보행 환경 (순수 환경은 'Link01Walk-v0')
저장폴더 = 'my_walk'
os.makedirs(저장폴더, exist_ok=True)

총학습걸음 = 1_000_000  # 빠른 확인은 200_000, 마스터 보행은 1_000_000 ~ 2_000_000
세상수 = 8             # 병렬 환경 수
학습률 = 3e-4
클립 = 0.2

# 2) 병렬 세상 만들기 및 관측치 51차원 정규화
venv = make_vec_env(f'link01_gym:{환경이름}', n_envs=세상수, vec_env_cls=SubprocVecEnv)
venv = VecNormalize(venv, norm_obs=True, norm_reward=False, clip_obs=10.0)

# 3) PPO 두뇌 구조 정의 (Actor: 64x64 Tanh -> 웹 플레이그라운드 규격 100% 호환)
policy_kwargs = dict(
    net_arch=dict(pi=[64, 64], vf=[256, 256]),
    activation_fn=torch.nn.Tanh,
)

model = PPO(
    'MlpPolicy',
    venv,
    policy_kwargs=policy_kwargs,
    learning_rate=학습률,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=클립,
    ent_coef=0.005,
    verbose=1,
)

# 4) 학습 시작
시작시각 = time.time()
print(f'🚀 [LINK-01 PPO 학습 시작] 총 {총학습걸음:,} 걸음 (병렬 환경: {세상수}개)...')
model.learn(total_timesteps=총학습걸음)
걸린시간_초 = time.time() - 시작시각
걸린분 = round(걸린시간_초 / 60, 2)
print(f'✅ 학습 완료! 소요 시간: {걸린분}분 ({걸린시간_초:.1f}초)')

# 5) 가중치 및 정규화 파라미터 저장
모델파일 = os.path.join(저장폴더, 'model.zip')
정규화파일 = os.path.join(저장폴더, 'vecnormalize.pkl')
model.save(모델파일)
venv.save(정규화파일)

# 6) CELL 06, 07 필수 연동 3요소 완성
# 단일 평가 환경을 이용한 결정론적 행동 추론 함수
행동 = lambda obs: model.predict(venv.normalize_obs(obs[None]), deterministic=True)[0][0]
모델파일들 = [모델파일, 정규화파일]
학습정보 = dict(steps=int(model.num_timesteps), minutes=걸린분, lr=학습률, clip=클립, env=환경이름)

print('📁 저장된 모델 파일들:', 모델파일들)
print('📊 학습 정보:', 학습정보)


## CELL 06 · 시험 + 리플레이 영상

처음 보는 씨앗 5판 × 20초, 명령 0.18 m/s. 넘어지면 다시 세우고 시간은 계속 흐릅니다.

In [ ]:
assert 행동 is not None, '먼저 CELL 05 를 채우고 실행하세요 (행동 · 모델파일들 · 학습정보)'
시험 = evaluate(행동, 'runs/my_walk_eval', episodes=5, seconds=20, label=f"LINK-01 · PPO · {학습정보.get('steps', 0):,} 걸음")
print(f"거리 {시험['distance_m_per_episode']} m / 20초 · 속도 {시험['speed_mps']} m/s · 넘어짐 {시험['falls_per_min']} 회/분")
show('runs/my_walk_eval/replay.mp4')

## CELL 07 · 🤗 허깅페이스에 올리기 — 6강과 같은 규칙

1. 코랩 왼쪽 **🔑 Secrets** 에 `HF_TOKEN` (Write 토큰)을 넣거나, 실행하면 나오는 입력창에 붙여넣으세요.
2. 저장소 이름은 알아서 **`connectailab-rl-이름-버전-link01-walk-ppo`** — 앞부분 `connectailab-rl` 은 꼭 그대로.
3. 올라가는 것: 내 두뇌 파일들 · `replay.mp4` · `connectailab.json`(시험 숫자 · 학습 정보 · 소개) · `README.md`(모델 카드)

> ⚠️ 토큰은 비밀번호와 같습니다. 코드나 파일에 적지 말고, 화면에 보이게 붙여넣지 마세요.

In [ ]:
이름 = ""             #@param {type:"string"}
버전 = "v1"           #@param {type:"string"}
나를_알리는_글 = """
"""
import hub_push
assert 모델파일들 and all(os.path.exists(f) for f in 모델파일들), '모델파일들 경로를 확인하세요 (CELL 05)'
# stable-baselines3 로 model · venv 이름을 썼다면 웹용 두뇌(actor.json)도 같이 만듭니다
if 'model' in globals() and 'venv' in globals():
    문제 = hub_push.export_actor_sb3(model, venv, 'my_walk/actor.json')
    print('actor.json', '웹 규칙 통과' if not 문제 else 문제)
    if not 문제 and 'my_walk/actor.json' not in 모델파일들: 모델파일들 = 모델파일들 + ['my_walk/actor.json']
폴더, 저장소 = hub_push.build(모델파일들, 시험, 학습정보, 이름 or 'me', 버전, 나를_알리는_글,
                          replay='runs/my_walk_eval/replay.mp4', out_dir='hf_upload')
print('올릴 폴더:', 폴더, sorted(os.listdir(폴더)))
올린주소 = hub_push.push(폴더, 저장소)          # 토큰은 여기서만 받습니다 (Secrets → 입력창)

> 🔒 이 노트북과 코드는 Connect AI LAB 이 만들었습니다. 공부에는 마음껏 쓰시고, 다른 강의를 만드는 데는 쓰지 말아 주세요.